The Dicom images provided in the dataset are just one slice each and contain no 3d information. Thus, I was unable to do the model assigned to me and instead just performed some preprocessing and normal CNN image classification. 

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import os

In [2]:
torch.cuda.is_available()

True

In [3]:
loaded_data = torch.load('../../dataset/processed_dataset.pt')
loaded_features = loaded_data['features']
loaded_labels = loaded_data['labels']
loaded_labels = loaded_labels[:,0]

In [4]:
class XrayImageDataset(torch.utils.data.Dataset):
    def __init__(self, data, labels):
        self.data = data
        self.labels = labels

    def __len__(self):
        return self.data.shape[0]

    def __getitem__(self, idx):
        return self.data[idx].unsqueeze(0), self.labels[idx]

In [5]:
wholedataset = XrayImageDataset(loaded_features, loaded_labels)

In [6]:
from torch.utils.data import DataLoader

In [7]:
train_len = int(len(wholedataset) * 0.8)  # 80% for training
val_len = int(len(wholedataset) * 0.1)    # 10% for validation
test_len = len(wholedataset) - train_len - val_len # Remaining 10% for testing
train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(wholedataset, [train_len, val_len, test_len])

In [8]:
train_len

1390

In [9]:
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=True)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(32 * 56 * 56, 128)  # depends on input size
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # (B,16,H/2,W/2)
        x = self.pool(F.relu(self.conv2(x)))  # (B,32,H/4,W/4)

        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SimpleCNN(num_classes=22).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [16]:
loaded_features[0].shape

torch.Size([224, 224])

In [17]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()

        outputs = model(x)
        loss = criterion(outputs, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)
            loss = criterion(outputs, y)

            total_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    acc = correct / total
    return total_loss / len(loader), acc

In [19]:
import time

In [21]:
train_loss, train_acc = validate(model, train_loader, criterion, device)

torch.Size([8, 1, 224, 224])


ZeroDivisionError: division by zero

In [15]:
num_epochs = 10

for epoch in range(num_epochs):
    start = time.time()
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    train_loss, train_acc = validate(model, train_loader, criterion, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)

    print(f"Epoch {epoch+1}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    print(f"Time elapsed: {time.time() - start:.2f}s")

Epoch 1
Train Loss: 1.3694
Train Loss: 1.3694, Train Acc: 0.6216
Val Loss: 1.6584, Val Acc: 0.5145
Time elapsed: 2.68s
Epoch 2
Train Loss: 0.9103
Train Loss: 0.9103, Train Acc: 0.7446
Val Loss: 1.4321, Val Acc: 0.6069
Time elapsed: 1.75s
Epoch 3
Train Loss: 0.5555
Train Loss: 0.5555, Train Acc: 0.8489
Val Loss: 1.3627, Val Acc: 0.5896
Time elapsed: 1.85s
Epoch 4
Train Loss: 0.1929
Train Loss: 0.1929, Train Acc: 0.9604
Val Loss: 1.4174, Val Acc: 0.6185
Time elapsed: 2.08s
Epoch 5
Train Loss: 0.0560
Train Loss: 0.0560, Train Acc: 0.9899
Val Loss: 1.4658, Val Acc: 0.6243
Time elapsed: 1.95s
Epoch 6
Train Loss: 0.0364
Train Loss: 0.0364, Train Acc: 0.9942
Val Loss: 1.5938, Val Acc: 0.6301
Time elapsed: 2.02s
Epoch 7
Train Loss: 0.1004
Train Loss: 0.1004, Train Acc: 0.9748
Val Loss: 1.9397, Val Acc: 0.6474
Time elapsed: 2.05s
Epoch 8
Train Loss: 0.0226
Train Loss: 0.0226, Train Acc: 0.9964
Val Loss: 1.6071, Val Acc: 0.6532
Time elapsed: 2.02s
Epoch 9
Train Loss: 0.0435
Train Loss: 0.0435, T

In [17]:
torch.save(model.state_dict(), 'weights.pth')

In [16]:
def train_model(model, num_epochs, device):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    for epoch in range(num_epochs):
        start = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        train_loss, train_acc = validate(model, train_loader, criterion, device)
        val_loss, val_acc = validate(model, val_loader, criterion, device)
    
        print(f"Epoch {epoch+1}")
        print(f"Train Loss: {train_loss:.4f}")
        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
        print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
        print(f"Time elapsed: {time.time() - start:.2f}s")

In [17]:
# --- SE Block ---
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.fc1 = nn.Linear(channels, channels // reduction)
        self.fc2 = nn.Linear(channels // reduction, channels)

    def forward(self, x):
        b, c, h, w = x.size()
        y = F.adaptive_avg_pool2d(x, 1).view(b, c)
        y = F.silu(self.fc1(y))
        y = torch.sigmoid(self.fc2(y)).view(b, c, 1, 1)
        return x * y


# --- Residual Block ---
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, drop_prob=0.0):
        super().__init__()

        # Use GroupNorm instead of BatchNorm
        def gn(channels):
            return nn.GroupNorm(num_groups=8, num_channels=channels)

        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.gn1 = gn(out_channels)

        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False)
        self.gn2 = gn(out_channels)

        self.se = SEBlock(out_channels)
        self.drop_prob = drop_prob

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride, bias=False),
                gn(out_channels)
            )

    def forward(self, x):
        identity = self.shortcut(x)

        out = F.silu(self.gn1(self.conv1(x)))
        out = self.gn2(self.conv2(out))
        out = self.se(out)

        # stochastic depth
        if self.training and torch.rand(1).item() < self.drop_prob:
            out = torch.zeros_like(out)

        out += identity
        return F.silu(out)


# --- Full Model ---
class ImprovedCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        def gn(channels):
            return nn.GroupNorm(num_groups=8, num_channels=channels)

        # ✅ 1-channel input
        self.stem = nn.Sequential(
            nn.Conv2d(1, 64, 3, 1, 1, bias=False),
            gn(64),
            nn.SiLU()
        )

        self.layer1 = self._make_layer(64, 64, 2, stride=1, drop_prob=0.05)
        self.layer2 = self._make_layer(64, 128, 2, stride=2, drop_prob=0.1)
        self.layer3 = self._make_layer(128, 256, 2, stride=2, drop_prob=0.15)

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def _make_layer(self, in_c, out_c, blocks, stride, drop_prob):
        layers = []
        layers.append(ResidualBlock(in_c, out_c, stride, drop_prob))
        for _ in range(1, blocks):
            layers.append(ResidualBlock(out_c, out_c, 1, drop_prob))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.pool(x)
        return self.fc(x)

In [18]:
model = ImprovedCNN(num_classes=22)

In [19]:
train_model(model, 20, device)

Epoch 1
Train Loss: 2.2562
Train Loss: 2.2562, Train Acc: 0.4158
Val Loss: 2.2800, Val Acc: 0.3988
Time elapsed: 32.05s
Epoch 2
Train Loss: 2.0311
Train Loss: 2.0311, Train Acc: 0.4252
Val Loss: 2.0563, Val Acc: 0.3931
Time elapsed: 31.65s
Epoch 3
Train Loss: 1.9895
Train Loss: 1.9895, Train Acc: 0.4410
Val Loss: 2.0545, Val Acc: 0.4162
Time elapsed: 30.93s
Epoch 4
Train Loss: 1.8093
Train Loss: 1.8093, Train Acc: 0.4597
Val Loss: 1.9315, Val Acc: 0.4393
Time elapsed: 32.29s
Epoch 5
Train Loss: 1.6915
Train Loss: 1.6915, Train Acc: 0.5259
Val Loss: 1.8174, Val Acc: 0.4913
Time elapsed: 32.75s
Epoch 6
Train Loss: 1.6187
Train Loss: 1.6187, Train Acc: 0.5540
Val Loss: 1.7522, Val Acc: 0.4971
Time elapsed: 31.65s
Epoch 7
Train Loss: 1.5899
Train Loss: 1.5899, Train Acc: 0.5540
Val Loss: 1.7132, Val Acc: 0.4855
Time elapsed: 32.00s
Epoch 8
Train Loss: 1.7273
Train Loss: 1.7273, Train Acc: 0.4935
Val Loss: 1.7644, Val Acc: 0.4798
Time elapsed: 32.65s
Epoch 9
Train Loss: 1.6646
Train Loss: 1

KeyboardInterrupt: 

In [43]:
def improved_train_model(model, num_epochs, device, criterion, optimizer, scheduler=None):
    model = model.to(device)
    for epoch in range(num_epochs):
        start = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        train_loss, train_acc = validate(model, train_loader, criterion, device)
        val_loss, val_acc = validate(model, val_loader, criterion, device)
        if scheduler:
            scheduler.step()
        print(f"Epoch {epoch+1}")
        print(f"Train Loss: {train_loss:.4f}")
        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
        print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
        print(f"Time elapsed: {time.time() - start:.2f}s")

In [22]:
from torch.optim.lr_scheduler import CosineAnnealingLR
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SimpleCNN(num_classes=22).to(device)
num_epochs=50
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs)

In [23]:
improved_train_model(model, num_epochs, device, criterion, optimizer, scheduler)

Epoch 1
Train Loss: 1.5765
Train Loss: 1.5765, Train Acc: 0.6698
Val Loss: 1.7464, Val Acc: 0.5954
Time elapsed: 2.35s
Epoch 2
Train Loss: 1.3352
Train Loss: 1.3352, Train Acc: 0.7554
Val Loss: 1.5790, Val Acc: 0.6416
Time elapsed: 2.12s
Epoch 3
Train Loss: 1.1561
Train Loss: 1.1561, Train Acc: 0.8388
Val Loss: 1.5624, Val Acc: 0.6647
Time elapsed: 2.01s
Epoch 4
Train Loss: 0.9426
Train Loss: 0.9426, Train Acc: 0.9173
Val Loss: 1.4570, Val Acc: 0.7168
Time elapsed: 2.02s
Epoch 5
Train Loss: 0.8048
Train Loss: 0.8048, Train Acc: 0.9806
Val Loss: 1.4561, Val Acc: 0.7052
Time elapsed: 1.97s
Epoch 6
Train Loss: 0.6996
Train Loss: 0.6996, Train Acc: 0.9964
Val Loss: 1.3726, Val Acc: 0.7110
Time elapsed: 2.05s
Epoch 7
Train Loss: 0.6764
Train Loss: 0.6764, Train Acc: 0.9978
Val Loss: 1.3628, Val Acc: 0.7168
Time elapsed: 2.07s
Epoch 8
Train Loss: 0.6570
Train Loss: 0.6570, Train Acc: 0.9986
Val Loss: 1.3360, Val Acc: 0.7283
Time elapsed: 2.01s
Epoch 9
Train Loss: 0.6504
Train Loss: 0.6504, T

KeyboardInterrupt: 

We see overfitting (high consistent train accuracy but validation accuracy drops and fluxuates wildly after more training). 
Apply techniques to deal with overfitting.

In [24]:
import torchvision.transforms as T

train_transforms = T.Compose([
    T.ToPILImage(),
    T.RandomResizedCrop(224, scale=(0.8, 1.0)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.ToTensor(),
    T.Normalize(mean=[0.5], std=[0.5])
])

In [25]:
class XrayImageDatasetWithTransforms(torch.utils.data.Dataset):
    def __init__(self, data, labels, transform=None):
        self.data = data
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return self.data.shape[0]

    def __getitem__(self, idx):
        x = self.data[idx].unsqueeze(0)
        y = self.labels[idx]
        
        if self.transform:
            x = self.transform(x)
        return x, y

In [36]:
train_dataset_with_transform = XrayImageDatasetWithTransforms(
    wholedataset.data[train_dataset.indices],
    wholedataset.labels[train_dataset.indices],
    transform=train_transforms
)

In [37]:
train_loader = DataLoader(train_dataset_with_transform, batch_size=8, shuffle=True)

In [38]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SimpleCNN(num_classes=22).to(device)
num_epochs=20
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs)

In [39]:
improved_train_model(model, num_epochs, device, criterion, optimizer, scheduler)


Epoch 1
Train Loss: 1.8802
Train Loss: 1.8802, Train Acc: 0.5388
Val Loss: 3.9210, Val Acc: 0.0347
Time elapsed: 8.39s
Epoch 2
Train Loss: 1.6973
Train Loss: 1.6973, Train Acc: 0.6245
Val Loss: 3.8924, Val Acc: 0.0231
Time elapsed: 8.95s
Epoch 3
Train Loss: 1.5923
Train Loss: 1.5923, Train Acc: 0.6568
Val Loss: 3.6354, Val Acc: 0.0405
Time elapsed: 8.81s
Epoch 4
Train Loss: 1.5447
Train Loss: 1.5447, Train Acc: 0.6791
Val Loss: 3.8631, Val Acc: 0.0289
Time elapsed: 9.13s
Epoch 5
Train Loss: 1.4609
Train Loss: 1.4609, Train Acc: 0.7043
Val Loss: 3.4297, Val Acc: 0.0462
Time elapsed: 8.86s
Epoch 6
Train Loss: 1.4641
Train Loss: 1.4641, Train Acc: 0.7180
Val Loss: 3.3812, Val Acc: 0.0462
Time elapsed: 9.13s
Epoch 7
Train Loss: 1.4021
Train Loss: 1.4021, Train Acc: 0.7295
Val Loss: 3.1430, Val Acc: 0.0925
Time elapsed: 9.38s
Epoch 8
Train Loss: 1.3989
Train Loss: 1.3989, Train Acc: 0.7309
Val Loss: 3.0639, Val Acc: 0.1040
Time elapsed: 9.05s
Epoch 9
Train Loss: 1.3718
Train Loss: 1.3718, T

Significant drop in performance for both training (expected) and validation (unexpected). 
Looking at the following data augmentation transforms we look into their affects:\
T.ToPILImage() - performance hit,\
T.RandomResizedCrop(224, scale=(0.8, 1.0)), - Random scaling, should be fine, not affect. \
T.RandomHorizontalFlip(), - Bad. What if it classifies left as right and right as left. causes confusion. \
T.RandomRotation(10), - 10 degrees of rotation shouldnt be too bad. However, orientation of the body should be set before xray operations start. Bad. \
T.ToTensor(), - Performance hit, already tensors. \
T.Normalize(mean=[0.5], std=[0.5]) - Could this also be causing issues.
\
Or could it be the updated cosine annealing?

In [40]:
train_transforms = T.Compose([
    # T.ToPILImage(),
    T.RandomResizedCrop(224, scale=(0.9, 1.0)) # Tighten range of scale
    # T.ToTensor(), - Already tensor
    # T.Normalize(mean=[0.5], std=[0.5])
])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SimpleCNN(num_classes=22).to(device)
num_epochs=20
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs)
improved_train_model(model, num_epochs, device, criterion, optimizer, scheduler)

Epoch 1
Train Loss: 1.7208
Train Loss: 1.7208, Train Acc: 0.6216
Val Loss: 3.1591, Val Acc: 0.0694
Time elapsed: 8.74s
Epoch 2
Train Loss: 1.5660
Train Loss: 1.5660, Train Acc: 0.6612
Val Loss: 2.9555, Val Acc: 0.1272
Time elapsed: 8.55s
Epoch 3
Train Loss: 1.4622
Train Loss: 1.4622, Train Acc: 0.7036
Val Loss: 3.6318, Val Acc: 0.0405
Time elapsed: 9.37s
Epoch 4
Train Loss: 1.3446
Train Loss: 1.3446, Train Acc: 0.7612
Val Loss: 3.6279, Val Acc: 0.0520
Time elapsed: 8.95s
Epoch 5
Train Loss: 1.2798
Train Loss: 1.2798, Train Acc: 0.7734
Val Loss: 3.7493, Val Acc: 0.1272
Time elapsed: 9.07s
Epoch 6
Train Loss: 1.2267
Train Loss: 1.2267, Train Acc: 0.7885
Val Loss: 3.6889, Val Acc: 0.1618
Time elapsed: 9.15s
Epoch 7
Train Loss: 1.2413
Train Loss: 1.2413, Train Acc: 0.7978
Val Loss: 3.8016, Val Acc: 0.1618
Time elapsed: 9.14s
Epoch 8
Train Loss: 1.1479
Train Loss: 1.1479, Train Acc: 0.8180
Val Loss: 3.5174, Val Acc: 0.2486
Time elapsed: 9.03s
Epoch 9
Train Loss: 1.1041
Train Loss: 1.1041, T

In [45]:
train_transforms = T.Compose([
    # T.ToPILImage(),
    # T.RandomResizedCrop(224, scale=(0.9, 1.0)) # Tighten range of scale
    # T.ToTensor(), - Already tensor
    # T.Normalize(mean=[0.5], std=[0.5])
])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SimpleCNN(num_classes=22).to(device)
num_epochs=20
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
# scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs)
improved_train_model(model, num_epochs, device, criterion, optimizer)

Epoch 1
Train Loss: 1.4160
Train Loss: 1.4160, Train Acc: 0.6281
Val Loss: 3.1070, Val Acc: 0.1098
Time elapsed: 8.52s
Epoch 2
Train Loss: 1.1357
Train Loss: 1.1357, Train Acc: 0.6871
Val Loss: 3.6036, Val Acc: 0.0636
Time elapsed: 9.18s
Epoch 3
Train Loss: 1.0623
Train Loss: 1.0623, Train Acc: 0.6942
Val Loss: 3.7158, Val Acc: 0.1561
Time elapsed: 8.75s
Epoch 4
Train Loss: 0.8868
Train Loss: 0.8868, Train Acc: 0.7201
Val Loss: 5.5298, Val Acc: 0.0751
Time elapsed: 9.07s
Epoch 5
Train Loss: 0.8231
Train Loss: 0.8231, Train Acc: 0.7597
Val Loss: 6.3319, Val Acc: 0.0983
Time elapsed: 9.23s


KeyboardInterrupt: 